In [ ]:
from pathlib import Path
import platform

REPO = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "dpsim-fmi/CMakeLists.txt").is_file()
    ),
    None,
)
if REPO is None:
    raise RuntimeError("Start the kernel inside the DPsim repository.")
BUILD_ROOT = REPO / (
    "build-fmi" if platform.system() == "Linux" else "build-macos-check"
)
CASE = REPO / "dpsim-fmi/examples/synchronous_generator"
BUILD = BUILD_ROOT / "fmi-examples/synchronous_generator"
BUILD.mkdir(parents=True, exist_ok=True)
(BUILD / "matlab-origin").mkdir(exist_ok=True)
for folder in [CASE / "sources", CASE / "matlab"]:
    for item in folder.iterdir():
        link = BUILD / "matlab-origin" / item.name
        if item.is_file() and not link.exists():
            link.symlink_to(item)
if not (BUILD / "matlab").exists():
    (BUILD / "matlab").symlink_to(CASE / "matlab", target_is_directory=True)

In [ ]:
# SPDX-License-Identifier: MPL-2.0
"""Validate nonlinear SG runs against MATLAB and timestep/tolerance checks.

Usage: python compare_nonlinear_sg.py dpsim-fmi/examples/synchronous_generator/results
Requires numpy/pandas. Engineering bases come from MATLAB's saved configuration;
no generator equations or fit parameters are used here.
"""
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd


def read(path):
    frame = pd.read_csv(path, skipinitialspace=True)
    frame.columns = frame.columns.str.strip()
    if "input_vref_pu" in frame and "vref_pu" not in frame:
        frame["vref_pu"] = frame["input_vref_pu"]
    assert len(frame) > 1 and np.isfinite(frame.to_numpy()).all(), path
    return frame


def signal_base(name, config):
    bases = config["bases"]
    if name in ("i_a", "i_b", "i_c") or name.startswith("i_line_"):
        return bases["phase_current_A"]
    if name.startswith("v_bus_"):
        return bases["phase_voltage_V"]
    if name.startswith("p_"):
        return bases["power_W"]
    if name == "field_current_A":
        return bases["field_current_A"]
    if name == "psi_f_Wb":
        return bases["field_flux_Wb"]
    if name.startswith("psi_"):
        return bases["stator_flux_Wb"]
    if name in ("theta_e_rad", "theta_state_rad"):
        return bases["angle_rad"]
    if name == "g_load_S":
        return bases["conductance_S"]
    return bases["per_unit"]


def compare(actual, reference, config):
    ta = actual["time" if "time" in actual else "time_s"].to_numpy()
    tr = reference["time" if "time" in reference else "time_s"].to_numpy()
    assert len(ta) == len(tr) and np.max(np.abs(ta - tr)) < 1e-9
    result = {}
    for name in reference.columns:
        if (
            name
            in (
                "time",
                "time_s",
                "newton_iterations",
                "newton_residual",
                "scaled_residual",
            )
            or name not in actual
        ):
            continue
        # Reference outputs comprise physical states, control diagnostics and
        # external grid signals. Importer-only bookkeeping is excluded.
        if name.startswith("network_current_") or name.startswith("input_"):
            continue
        error = actual[name].to_numpy() - reference[name].to_numpy()
        base = signal_base(name, config)
        maximum = float(np.max(np.abs(error)))
        rmse = float(np.sqrt(np.mean(error**2)))
        result[name] = dict(
            max_abs=maximum,
            rmse=rmse,
            engineering_base=base,
            max_normalized=maximum / base,
            rms_normalized=rmse / base,
        )
    assert len(result) == 34, "all 34 benchmark signals must be compared"
    return result


def validate(directory):
    directory = Path(directory)
    config = json.loads((directory / "matlab/benchmark.json").read_text())
    matlab = {
        step: read(directory / f"matlab/reference_trapezoidal_{step}us.csv")
        for step in (50, 25)
    }
    actual = {
        step: read(directory / f"dpsim-{step}us/logs/fmi_sg_nonlinear.csv")
        for step in (50, 25)
    }
    tight = read(directory / "dpsim-50us-tight/logs/fmi_sg_nonlinear.csv")
    ode = read(directory / "matlab-origin/reference_matlab_ode15s.csv")
    report = {
        "acceptance_max_and_rms_normalized": 1e-3,
        "engineering_bases": config["bases"],
        "logger_note": "DPsim CSV rounds signal values to six decimals; small errors can be quantization-limited.",
    }
    for step in (50, 25):
        report[f"dpsim_vs_trapezoidal_{step}us"] = compare(
            actual[step], matlab[step], config
        )
        for metric in report[f"dpsim_vs_trapezoidal_{step}us"].values():
            assert metric["max_normalized"] < 1e-3 and metric["rms_normalized"] < 1e-3
        frame = actual[step]
        assert len(frame) == round(config["stop_time_s"] / (step * 1e-6)) + 1
        assert frame.newton_residual.max() <= 1
        assert frame.newton_iterations.iloc[1:].min() >= 1
        residual = max(
            float(np.max(np.abs(frame[f"i_{p}"] - frame[f"network_current_{k}"])))
            for k, p in enumerate("abc")
        )
        assert residual < 1.1e-6
        report[f"solver_{step}us"] = dict(
            max_scaled_residual=float(frame.newton_residual.max()),
            max_newton_iterations=int(frame.newton_iterations.max()),
            max_kcl_residual_A=residual,
        )
    report["tight_vs_default_tolerance"] = compare(tight, actual[50], config)
    # Both steps are compared to ode15s on the common original 50-us grid,
    # so no linear interpolation of the sharp switching transient is needed.
    report["trapezoidal_vs_ode15s_50us"] = compare(matlab[50], ode, config)
    report["trapezoidal_vs_ode15s_25us"] = compare(
        matlab[25].iloc[::2].reset_index(drop=True), ode, config
    )
    for step in (50, 25):
        for metric in report[f"trapezoidal_vs_ode15s_{step}us"].values():
            assert metric["max_normalized"] < 1e-3 and metric["rms_normalized"] < 1e-3
    report["bus_voltage_refinement_ratio"] = (
        report["trapezoidal_vs_ode15s_50us"]["v_bus_a_V"]["max_abs"]
        / report["trapezoidal_vs_ode15s_25us"]["v_bus_a_V"]["max_abs"]
    )
    assert 3.5 < report["bus_voltage_refinement_ratio"] < 4.5
    (directory / "comparison.json").write_text(json.dumps(report, indent=2))
    return report


from types import SimpleNamespace

comparison = SimpleNamespace(validate=validate, read=read)

# Nonlinear three-phase FMU inside DPsim MNA

The ten-state MATLAB generator, AVR and governor remain inside a persistent FMI 3.0 **Model Exchange** instance. DPsim converges implicit trapezoidal state residuals and network KCL together using a condensed Norton Jacobian. The surrounding six-state grid is a separate current-input SSN branch. The C++ importer contains no generator equations.

The independent MATLAB trapezoidal reference solves all sixteen states in one Newton system, using complex-step Jacobians. This differs from DPsim's component condensation and finite-difference FMI Jacobians. Both use identical initialization and events at 1 s (load +6 kW nominal) and 2 s (voltage reference +2%). The original tighter ode15s run is a third comparison. All 34 physical/control/reference signals are checked against declared engineering bases, with maximum and RMS normalized errors required below 0.1%.

Use the **DPsim Python** kernel and **Run All**. This reruns the C++ simulations; MATLAB references must already exist. Saving the executed notebook embeds its plots.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = BUILD
FMU_NAME = (
    "dpsim_sg_3ph_me.x86_64-linux.fmu"
    if platform.system() == "Linux"
    else "dpsim_sg_3ph_me.fmu"
)
FMU = CASE / "sources" / FMU_NAME
CONFIG = ROOT / "matlab/benchmark.json"
EXE = BUILD_ROOT / "dpsim-fmi/FMI_EMT_Ph3_Nonlinear"
PLOTS = ROOT / "comparison"
PLOTS.mkdir(parents=True, exist_ok=True)
for path in (
    FMU,
    CONFIG,
    EXE,
    ROOT / "matlab/reference_trapezoidal_50us.csv",
    ROOT / "matlab/reference_trapezoidal_25us.csv",
):
    assert path.is_file(), f"Missing {path}"
config = json.loads(CONFIG.read_text())
print("FMU:", FMU)
print("Engineering bases:", config["bases"])

In [ ]:
import os

for name, step, factor in [
    ("dpsim-50us", 5e-5, 1),
    ("dpsim-25us", 2.5e-5, 1),
    ("dpsim-50us-tight", 5e-5, 0.1),
]:
    directory = ROOT / name
    directory.mkdir(parents=True, exist_ok=True)
    (ROOT / "tmp").mkdir(exist_ok=True)
    env = dict(os.environ, TMPDIR=str(ROOT / "tmp"))
    result = subprocess.run(
        [str(EXE), str(FMU), str(CONFIG), str(step), str(factor)],
        cwd=directory,
        env=env,
        text=True,
        capture_output=True,
    )
    (directory / "run.log").write_text(result.stdout + result.stderr)
    assert result.returncode == 0, result.stdout + result.stderr
    print(name, result.stdout.splitlines()[-1])

## Acceptance, convergence and refinement

The checks cover every saved state/diagnostic, precise event timestamps, per-phase KCL and converged Newton residuals. The phase voltage/current bases are peak quantities derived from the declared nominal voltage and power. Angle uses a 1 rad base; per-unit signals use 1 pu. Full bases and absolute errors are saved in `comparison.json`.

The standard DPsim CSV logger rounds signals to six decimal places. Differences at that level, including between default and tighter Newton settings, cannot be resolved by the saved CSV. The normalized Newton residual is separately checked before every state commit.

In [ ]:
report = comparison.validate(ROOT)
rows = []
for case, signals in report.items():
    if isinstance(signals, dict) and "i_a" in signals:
        rows.extend(
            dict(case=case, signal=name, **values) for name, values in signals.items()
        )
metrics = pd.DataFrame(rows)
metrics.to_csv(PLOTS / "metrics.csv", index=False)
display(metrics.groupby("case")[["max_normalized", "rms_normalized"]].max())
print("Newton/KCL 50 us:", report["solver_50us"])
print("Newton/KCL 25 us:", report["solver_25us"])
print("Voltage error refinement ratio:", report["bus_voltage_refinement_ratio"])
a = comparison.read(ROOT / "dpsim-50us/logs/fmi_sg_nonlinear.csv")
fine = comparison.read(ROOT / "dpsim-25us/logs/fmi_sg_nonlinear.csv")
b = comparison.read(ROOT / "matlab/reference_trapezoidal_50us.csv")
matfine = comparison.read(ROOT / "matlab/reference_trapezoidal_25us.csv")
ode = comparison.read(ROOT / "matlab-origin/reference_matlab_ode15s.csv")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
for p, phase in enumerate("abc"):
    for col, key, unit in [(0, f"i_{phase}", "A"), (1, f"v_bus_{phase}_V", "V")]:
        ax = axes[p, col]
        ax.plot(ode.time_s, ode[key], color="0.6", linewidth=2, label="MATLAB ode15s")
        ax.plot(b.time_s, b[key], "--", linewidth=1.3, label="MATLAB trapezoidal")
        ax.plot(a.time, a[key], linewidth=0.9, label="DPsim nonlinear FMU")
        ax.set(xlim=(0.98, 1.06), title=f"Phase {phase.upper()}: {key}", ylabel=unit)
        ax.grid(alpha=0.25)
        if p == 2:
            ax.set_xlabel("Time [s]")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Load-step EMT waveforms: nonlinear FMU solved inside DPsim MNA")
fig.tight_layout()
fig.savefig(PLOTS / "waveforms.png", dpi=170, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
w = 2 * np.pi * config["frequency_Hz"]
traces = [
    ("v_measured_pu", "Measured voltage", "pu"),
    ("speed_pu", "Rotor speed deviation", "Hz"),
    ("efd_pu", "Field supply", "pu"),
    ("pm_pu", "Prime-mover power", "pu"),
    ("theta_e_rad", "Rotor angle relative to grid", "deg"),
    ("p_gen_W", "Generator terminal power", "kW"),
]
for ax, (key, title, unit) in zip(axes.flat, traces):
    for frame, time, label, style in [
        (ode, ode.time_s, "MATLAB ode15s", "-"),
        (b, b.time_s, "MATLAB trapezoidal", "--"),
        (a, a.time, "DPsim nonlinear FMU", ":"),
    ]:
        values = frame[key]
        if key == "speed_pu":
            values = (values - 1) * config["frequency_Hz"]
        elif key == "theta_e_rad":
            values = (values - w * time + np.pi / 2) * 180 / np.pi
        elif key == "p_gen_W":
            values = values / 1000
        ax.plot(time, values, style, label=label, linewidth=1.2)
    if key == "v_measured_pu":
        ax.plot(b.time_s, b.vref_pu, color="black", alpha=0.4, label="Reference")
    ax.set(title=title, ylabel=unit, xlabel="Time [s]")
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS / "controls.png", dpi=170, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 8), sharex=True)
for ax, (key, unit) in zip(
    axes.flat,
    [
        ("i_a", "A"),
        ("v_bus_a_V", "V"),
        ("speed_pu", "pu"),
        ("efd_pu", "pu"),
        ("p_gen_W", "W"),
        ("psi_f_Wb", "Wb"),
    ],
):
    ax.plot(a.time, a[key] - b[key], linewidth=0.7)
    ax.set(
        title=f"DPsim minus MATLAB trapezoidal: {key}", ylabel=unit, xlabel="Time [s]"
    )
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS / "errors.png", dpi=170, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, key, unit in [(axes[0], "i_a", "A"), (axes[1], "v_bus_a_V", "V")]:
    ax.plot(b.time_s, b[key] - ode[key], label="50 us")
    ax.plot(b.time_s, matfine[key].to_numpy()[::2] - ode[key].to_numpy(), label="25 us")
    ax.set(
        xlim=(1, 1.03),
        title=f"Trapezoidal minus ode15s: {key}",
        xlabel="Time [s]",
        ylabel=unit,
    )
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(PLOTS / "refinement.png", dpi=170, bbox_inches="tight")
plt.show()

The matched trapezoidal comparison passes at both timesteps. Halving h reduces the bus-voltage discrepancy against ode15s by approximately four, consistent with second-order integration. Tightening Newton tolerances by ten does not change the six-decimal logged traces.

The machine/controller initial conditions are fixed to this solved operating point. Scheduled steps are split at their exact boundaries. Internal state events, state reinitialization, clocks and arbitrary discrete models remain explicit importer limitations; this benchmark does not establish support for them. MATLAB completed-step events that leave continuous states unchanged are handled.

The generated CSVs, plots, metrics and executed notebook are local files under the SSD development directory. The source notebook does not embed the external FMU/reference datasets.